# HistOrniGraph — New Corpus + Multimodal Corpus + Dedup (Colab)

Rebuilds the corpus cleanly from the **current** Drive data, builds the
**multimodal** catalogue, and runs duplicate detection — including an explicit
check for the **Vol 1 → Vol 15 cross-run contamination**.

Run cells top to bottom. Edit **§2 Config** first. Nothing is destructive:
the deduplicated corpus is written to a *new* directory and the originals on
Drive are never modified.


## 1 · Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Config — edit these

Set `ADDONS_ZIP` to wherever you uploaded `HistOrniGraph_addons_combined.zip`
on Drive. If you'd rather clone from git (after you've committed the add-ons),
see the commented block in §3.


In [ ]:
from pathlib import Path

# The uploaded zip on your Drive:
ADDONS_ZIP  = '/content/drive/MyDrive/HistOrniGraph_output/HistOrniGraph_addons_combined.zip'

# Root holding the Laubmann_NN_gemini/ folders (each with regions/*.json):
OUTPUT_BASE = Path('/content/drive/MyDrive/HistOrniGraph_output')

# Fresh dated corpus dir (keeps this run distinct from older corpora):
CORPUS_DIR  = OUTPUT_BASE / 'corpus_2026-07-21'

# Diary volumes only. Vol 35 is the general index/register — exclude it.
# Tip: test with a small set first, e.g. [1, 6, 15], then switch to 1..34.
VOLUMES = list(range(1, 35))     # 1..34

print('ADDONS_ZIP :', ADDONS_ZIP)
print('OUTPUT_BASE:', OUTPUT_BASE)
print('CORPUS_DIR :', CORPUS_DIR)
print('VOLUMES    :', VOLUMES[0], '..', VOLUMES[-1])

## 3 · Unpack the add-ons & install deps

In [ ]:
import os, sys

# Unpack the uploaded zip into /content
!unzip -o "{ADDONS_ZIP}" -d /content/ >/dev/null
ADDONS = '/content/HistOrniGraph_addons'

# --- OR, after you've committed the add-ons into the repo, clone instead: ---
# REPO='/content/HistOrniGraph'
# !git clone https://github.com/Maelkolb/HistOrniGraph.git {REPO} 2>/dev/null || (cd {REPO} && git pull --ff-only)
# ADDONS = f'{REPO}/HistOrniGraph_addons'

DEDUP = f'{ADDONS}/dedup'
assert os.path.isdir(f'{ADDONS}/laubmann_corpus'), 'laubmann_corpus package not found'
assert os.path.isfile(f'{DEDUP}/apply_dedup.py'), 'dedup tools not found'

# Dedup deps (corpus builder itself is stdlib-only):
!pip -q install rapidfuzz datasketch
!pip -q install imagehash pillow      # optional image-hash layer
print('addons unpacked at', ADDONS)

## 4 · Sanity check — the two toolsets talk to each other

Runs the dedup unit tests and confirms `apply_dedup.py` can import the entry
helpers from the refactored `laubmann_corpus` package (the seam between the two
pieces of work).


In [ ]:
!cd "{DEDUP}" && python test_laubmann_dedup.py

# Confirm the cross-package import resolves (entries regeneration depends on it):
import importlib, sys
sys.path.insert(0, ADDONS)
from laubmann_corpus.entries import find_entry_starts, strip_markup
print('cross-package import OK — apply_dedup will regenerate entries.*')

## 5 · Build the new corpus (text + multimodal + report, one pass)

`--multimodal` and `--report` **add** their outputs to the run (they are no
longer mutually exclusive). A **schema census** prints first as a sanity check.

Writes under `CORPUS_DIR`:
- `corpus.md` / `corpus.json` / `corpus.txt` / `entries.jsonl` / `entries.csv`
- `by_volume/Laubmann_NN.md`
- `multimodal/multimodal.{jsonl,csv,md}` — images, objects, sketches, marginalia, inserts (incl. folded)
- `report.{json,md}` — per-volume coverage


In [ ]:
VOLS_ARG = ' '.join(str(v) for v in VOLUMES) if VOLUMES else ''
vol_flag = f'--volumes {VOLS_ARG}' if VOLS_ARG else ''

!cd "{ADDONS}" && python build_corpus.py \
    --output-base "{OUTPUT_BASE}" \
    --corpus-dir  "{CORPUS_DIR}" \
    --per-volume --multimodal --report {vol_flag}

In [ ]:
!ls /content/drive/MyDrive/HistOrniGraph_output/corpus_2026-07-21

In [ ]:
!apt-get -qq install -y libpango-1.0-0 libpangocairo-1.0-0 libpangoft2-1.0-0 libgdk-pixbuf-2.0-0 libffi-dev libcairo2 fonts-dejavu >/dev/null
!pip -q install weasyprint pypdf
import os, re, html, json, csv, glob

# ----------------------------- CONFIG -----------------------------
CORPUS_DIR   = "/content/drive/MyDrive/HistOrniGraph_output/corpus_2026-07-21"
OUT_DIR      = None            # None -> write PDFs next to the corpus
TEXT_CORPUS  = None            # None -> auto-detect the main corpus .md
MULTIMODAL   = None            # None -> auto-detect *multimodal* file
VOLUMES      = None            # None -> all; or e.g. [1, 2, 3]
REFLOW       = True            # True: join OCR line-wraps into paragraphs
DEHYPHENATE  = True            # only used when REFLOW=True
SPLIT_PER_VOLUME = False       # True: one text PDF per volume instead of one big file
EMBED_CROPS  = False           # multimodal: embed the region crop images
HISTORNIGRAPH_ROOT = "/content/drive/MyDrive/HistOrniGraph_output"  # to resolve crops
TITLE        = "Laubmann — Ornithologische Tagebücher"
SUBTITLE     = "Digital Edition · HistOrniGraph"
# ------------------------------------------------------------------

RE_PAGE   = re.compile(r"^<!--\s*page\s+(.*?)\s*-->\s*$")
RE_REGION = re.compile(r"^<!--\s*region\s+(.*?)\s*-->\s*$")
RE_ATTR   = re.compile(r"(\w+)=(\S*)")
RE_VOL    = re.compile(r"^#\s+.*Vol\.\s*(\d+)", re.I)
RE_ENTRY  = re.compile(r"^\*\*\s*⮞?\s*Entry\s*[—-]\s*(.*?)\s*\*\*\s*$")
RE_ISO    = re.compile(r"`([^`]*)`\s*$")
ALLOWED_TAGS = ["u", "sup", "sub", "s", "b", "i", "em", "strong"]

def _attrs(s):
    return {k: v for k, v in RE_ATTR.findall(s)}

def _inline(text):
    text = html.escape(text, quote=False)
    for t in ALLOWED_TAGS:
        text = text.replace(f"&lt;{t}&gt;", f"<{t}>").replace(f"&lt;/{t}&gt;", f"</{t}>")
    text = re.sub(r"&lt;br\s*/?&gt;", "<br>", text)
    text = re.sub(r"\\([.\-_*`(){}\[\]#+!])", r"\1", text)
    text = re.sub(r"`([^`]+)`", r"<code>\1</code>", text)
    text = re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", text)
    text = re.sub(r"(?<!\*)\*(?!\*)(.+?)(?<!\*)\*(?!\*)", r"<em>\1</em>", text)
    return text

def _dehyphenate(lines):
    out, i = [], 0
    while i < len(lines):
        cur = lines[i]
        stripped = re.sub(r"(</[a-z]+>)\s*$", "", cur)
        if i + 1 < len(lines) and re.search(r"[A-Za-zÄÖÜäöüß]-$", stripped):
            nxt = lines[i + 1]
            joined = re.sub(r"-(\s*(?:</[a-z]+>)?)\s*$", r"\1", cur) + nxt.lstrip()
            joined = joined.replace("</u><u>", "").replace("</sup><sup>", "")
            lines[i + 1] = joined
            i += 1
            continue
        out.append(cur)
        i += 1
    return out

def _is_anno(ln):
    return bool(re.match(r"^\*\[.*\]\*$", ln.strip()))
def _is_cont(ln):
    return ln.strip().startswith("*(…continued") or "…continued from" in ln

def _emit_block(blk):
    if REFLOW:
        work = _dehyphenate(list(blk)) if DEHYPHENATE else list(blk)
        para = " ".join(x.strip() for x in work if x.strip())
        para = re.sub(r"\s+(\d+\.\))", r"<br>\1", para)
        return f"<p>{_inline(para)}</p>"
    return "<p class='diplo'>" + "<br>".join(_inline(x) for x in blk) + "</p>"

def _paragraphs(body):
    out, cur = [], []
    def flush():
        if cur: out.append(_emit_block(cur)); cur.clear()
    for ln in body:
        s = ln.strip()
        if s == "":
            flush()
        elif _is_anno(ln):
            flush(); out.append(f"<p class='anno'>{_inline(s.strip('*'))}</p>")
        elif _is_cont(ln):
            flush(); out.append(f"<div class='cont'>{_inline(s.strip('*'))}</div>")
        else:
            cur.append(ln.rstrip("\n"))
    flush()
    return "\n".join(out)

def _parse_table(body):
    rows = [r for r in body if r.strip().startswith("|")]
    cells = []
    for r in rows:
        if re.match(r"^\s*\|[\s:|-]+\|\s*$", r):
            continue
        parts = [c.strip() for c in r.strip().strip("|").split("|")]
        cells.append(parts)
    if not cells:
        return "<p>" + _inline(" ".join(body)) + "</p>"
    out = ["<table>"]
    for row in cells:
        out.append("<tr>" + "".join(f"<td>{_inline(c)}</td>" for c in row) + "</tr>")
    out.append("</table>")
    return "\n".join(out)

def _entry_header(line):
    m = RE_ENTRY.match(line)
    inner = m.group(1) if m else line.strip("* ")
    iso = ""
    mi = RE_ISO.search(inner)
    if mi:
        iso = mi.group(1).strip()
        inner = inner[:mi.start()].strip()
    parts = [p.strip() for p in inner.split("·")]
    date_txt = parts[0] if parts else inner
    place = " · ".join(parts[1:]) if len(parts) > 1 else ""
    badge = f"<span class='badge'>{html.escape(iso)}</span>" if iso else ""
    pl = f"<span class='place'>{_inline(place)}</span>" if place else ""
    return f"<div class='entry-h'>{badge}<span class='edate'>{_inline(date_txt)}</span>{pl}</div>"

def _region_html(rtype, body):
    if not body:
        return ""
    first = body[0].strip()
    head = ""
    rest = body
    if RE_ENTRY.match(first):
        head = _entry_header(first); rest = body[1:]
    elif first.startswith("*(…continued") or "…continued from" in first:
        head = f"<div class='cont'>{_inline(first.strip('*'))}</div>"; rest = body[1:]
    if rtype == "TableRegion":
        return head + _parse_table(rest)
    if rtype == "FootnoteRegion":
        return head + "<div class='fn'>" + _paragraphs(rest) + "</div>"
    cls = " class='listr'" if rtype == "ListRegion" else ""
    inner = _paragraphs(rest)
    return head + (f"<div{cls}>{inner}</div>" if cls else inner)

def iter_volumes(path, volumes=None):
    vol_num, rtype, body = None, None, []
    buf = []
    def open_page(meta):
        v = meta.get("volume", "")
        scan = meta.get("scan", "")
        pn = meta.get("page_number", "")
        label = f"Vol. {int(v):02d} · scan {int(scan):04d}" if v and scan else "page"
        if pn: label += f" · p. {pn}"
        buf.append(f"<div class='pdiv'><span class='pl'>{html.escape(label)}</span>"
                   f"<span class='pid'>{html.escape(meta.get('page_id',''))}</span></div>")

    with open(path, encoding="utf-8") as f:
        cur_vol_buf = None
        for raw in f:
            line = raw.rstrip("\n")
            mv = RE_VOL.match(line)
            if mv:
                if rtype is not None and body: buf.append(_region_html(rtype, body))
                body, rtype = [], None
                if vol_num is not None and (volumes is None or vol_num in volumes):
                    yield vol_num, cur_vol_buf, "".join(buf)
                vol_num = int(mv.group(1))
                buf = []
                cur_vol_buf = line[2:].strip()
                continue
            mp = RE_PAGE.match(line)
            if mp:
                if rtype is not None and body: buf.append(_region_html(rtype, body)); body, rtype = [], None
                open_page(_attrs(mp.group(1)))
                continue
            mr = RE_REGION.match(line)
            if mr:
                if rtype is not None and body: buf.append(_region_html(rtype, body))
                body = []
                rtype = _attrs(mr.group(1)).get("type", "ParagraphRegion")
                continue
            if line.startswith("## "):
                continue
            if line.startswith("# ") and not RE_VOL.match(line):
                if rtype is not None and body: buf.append(_region_html(rtype, body)); body, rtype = [], None
                buf.append(f"<h2 class='art'>{_inline(line[2:].strip())}</h2>")
                continue
            if re.match(r"^`[0-9a-f].*`$", line.strip()):
                continue
            if rtype is not None:
                body.append(line)
        if rtype is not None and body: buf.append(_region_html(rtype, body))
        if vol_num is not None and (volumes is None or vol_num in volumes):
            yield vol_num, cur_vol_buf, "".join(buf)

CSS = """
@page { size: A4; margin: 20mm 18mm 22mm 18mm;
  @top-center { content: string(voltitle); font: italic 8pt Georgia, serif; color:#8a7f6a; }
  @bottom-center { content: counter(page); font: 8pt Georgia, serif; color:#8a7f6a; } }
@page :first { @top-center { content: none } @bottom-center { content: none } }
* { box-sizing: border-box; }
body { font: 10.5pt/1.44 Georgia, "DejaVu Serif", "Liberation Serif", serif; color:#232019; }
.title-page { height: 250mm; display:flex; flex-direction:column; justify-content:center;
  align-items:center; text-align:center; page-break-after: always; }
.title-page h1 { font-size: 30pt; margin:0 0 6mm; letter-spacing:.5px; color:#3a2f22; }
.title-page .sub { font-style:italic; color:#8a7f6a; font-size:13pt; }
.title-page .rule { width:70mm; border-top:1px solid #c9bfa8; margin:9mm 0; }
.title-page .meta { color:#8a7f6a; font-size:9.5pt; margin-top:8mm; }
h1.vol { string-set: voltitle content(); page-break-before: always;
  font-size:19pt; color:#3a2f22; border-bottom:2px solid #b9853f;
  padding-bottom:3mm; margin:0 0 7mm; }
h2.art { font-size:13pt; color:#5a4a34; text-align:center; margin:8mm 0 3mm;
  break-after: avoid; font-variant: small-caps; letter-spacing:.5px; }
.pdiv { display:flex; justify-content:space-between; align-items:baseline;
  border-top:1px solid #e3dcc9; margin:7mm 0 2.5mm; padding-top:1.6mm;
  break-after: avoid; }
.pdiv .pl { font-variant: small-caps; letter-spacing:.6px; font-size:8.6pt; color:#9a7b3f; }
.pdiv .pid { font: 6.6pt "DejaVu Sans Mono", monospace; color:#c3b89f; }
.entry-h { break-after: avoid; margin:4.5mm 0 1.6mm; padding-left:3mm;
  border-left:3px solid #b9853f; line-height:1.3; }
.entry-h .badge { font:7.6pt "DejaVu Sans Mono", monospace; background:#f3ecdb;
  color:#8a6a2e; padding:.4mm 1.6mm; border-radius:2px; margin-right:2.5mm;
  vertical-align:1px; }
.entry-h .edate { font-weight:bold; color:#2c2418; }
.entry-h .place { color:#7a6a4a; font-style:italic; margin-left:2mm; }
.cont { font-style:italic; color:#9a8f78; font-size:9pt; margin:2mm 0 1mm; }
.anno { font-style:italic; color:#a99; font-size:8.6pt; }
p { margin:0 0 2mm; text-align:justify; hyphens:none; }
p.diplo { text-align:left; }
u { text-decoration: none; border-bottom:1px solid #cbb98f; }
code { font:8.4pt "DejaVu Sans Mono", monospace; color:#8a6a2e; }
.fn { font-size:8.8pt; color:#5c554770; color:#5f584a; border-top:1px solid #ece5d4;
  margin-top:2.5mm; padding-top:1.6mm; }
.fn p { text-align:left; }
.listr p { margin-bottom:1.4mm; }
table { border-collapse:collapse; width:100%; margin:2.5mm 0; font-size:9pt; }
td { border:1px solid #ddd4bf; padding:1mm 2mm; vertical-align:top; }
.mm-card { border:1px solid #e3dcc9; border-left:3px solid #7a9a6a; border-radius:3px;
  padding:3mm 4mm; margin:0 0 4mm; break-inside: avoid; }
.mm-card .mh { display:flex; justify-content:space-between; align-items:baseline; margin-bottom:1.5mm; }
.mm-card .mt { font-variant:small-caps; letter-spacing:.5px; color:#4f6a3f; font-size:9.5pt; }
.mm-card .mref { font:6.8pt "DejaVu Sans Mono", monospace; color:#b8b09a; }
.mm-card .chip { display:inline-block; font-size:7.4pt; background:#eef2e6; color:#4f6a3f;
  padding:.3mm 1.6mm; border-radius:2px; margin-right:1.5mm; }
.mm-card .vt { font-style:italic; color:#5f584a; border-left:2px solid #d9d2c0;
  padding-left:2.5mm; margin:1.5mm 0; }
.mm-card img { max-width:100%; max-height:70mm; display:block; margin:2mm auto 0;
  border:1px solid #e3dcc9; }
"""

def _doc(title_html, body_html):
    return (f"<html><head><meta charset='utf-8'><style>{CSS}</style></head>"
            f"<body>{title_html}{body_html}</body></html>")

def _title_page(sub=None):
    s = f"<div class='sub'>{html.escape(sub)}</div>" if sub else ""
    return (f"<div class='title-page'><h1>{html.escape(TITLE)}</h1>"
            f"<div class='rule'></div>{s}"
            f"<div class='meta'>{html.escape(SUBTITLE)}</div></div>")


def _write_pdf(html_str, out_path, base_url=None):
    from weasyprint import HTML
    HTML(string=html_str, base_url=base_url or os.getcwd()).write_pdf(out_path)

def _merge(pdf_paths, out_path):
    try:
        from pypdf import PdfWriter
    except Exception:
        from PyPDF2 import PdfWriter
    w = PdfWriter()
    for p in pdf_paths:
        w.append(p)
    with open(out_path, "wb") as fh:
        w.write(fh)

def render_text_pdf(text_path, out_path, volumes=None, keep_parts=False):
    tmp = out_path + ".parts"
    os.makedirs(tmp, exist_ok=True)
    title_pdf = os.path.join(tmp, "000_title.pdf")
    _write_pdf(_doc(_title_page(SUBTITLE), ""), title_pdf)
    parts = [title_pdf]
    for vnum, vtitle, vhtml in iter_volumes(text_path, volumes=volumes):
        vpdf = os.path.join(tmp, f"vol_{vnum:02d}.pdf")
        body = f"<h1 class='vol'>{html.escape(vtitle)}</h1>{vhtml}"
        _write_pdf(_doc("", body), vpdf)
        parts.append(vpdf)
        print(f"  rendered {vtitle}  ->  {os.path.getsize(vpdf)//1024} KB")
    _merge(parts, out_path)
    if not keep_parts:
        for p in parts: os.remove(p)
        os.rmdir(tmp)
    print(f"text PDF: {out_path}  ({os.path.getsize(out_path)//1024} KB)")
    return out_path

def _resolve_crop(row):
    crop = (row.get("crop") or "").strip()
    if not crop: return None
    vol = row.get("volume") or row.get("vol") or ""
    try: vdir = f"Laubmann_{int(vol):02d}_gemini"
    except Exception: return None
    p = os.path.join(HISTORNIGRAPH_ROOT, vdir, crop)
    return p if os.path.exists(p) else None

def _mm_card(row):
    typ = row.get("type", "Region")
    vol = row.get("volume") or row.get("vol") or ""
    scan = row.get("scan", ""); pn = row.get("page_number", "")
    try: ref = f"Vol. {int(vol):02d} · scan {int(scan):04d}"
    except Exception: ref = f"Vol. {vol} · scan {scan}"
    if pn: ref += f" · p. {pn}"
    ruid = row.get("region_uid", "")
    chips = []
    for k in ("drawing_type", "object_type", "insert_state", "page_side"):
        v = (row.get(k) or "").strip()
        if v: chips.append(f"<span class='chip'>{html.escape(k.split('_')[0])}: {html.escape(v)}</span>")
    chip_html = " ".join(chips)
    desc = (row.get("description") or "").strip()
    vt = (row.get("visible_text") or row.get("text") or "").strip()
    parts = [f"<div class='mh'><span class='mt'>{html.escape(typ)}</span>"
             f"<span class='mref'>{html.escape(ref)} · {html.escape(ruid)}</span></div>"]
    if chip_html: parts.append(f"<div>{chip_html}</div>")
    if desc: parts.append(f"<p>{_inline(desc)}</p>")
    if vt: parts.append(f"<div class='vt'>{_inline(vt)}</div>")
    if EMBED_CROPS:
        cp = _resolve_crop(row)
        if cp: parts.append(f"<img src='file://{cp}'>")
    return f"<div class='mm-card'>{''.join(parts)}</div>"

def _mm_rows(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in (".jsonl", ".ndjson"):
        return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    if ext == ".json":
        d = json.load(open(path, encoding="utf-8"))
        return d if isinstance(d, list) else d.get("rows", [])
    with open(path, encoding="utf-8") as fh:
        return list(csv.DictReader(fh))

def render_multimodal_pdf(mm_path, out_path):
    ext = os.path.splitext(mm_path)[1].lower()
    if ext in (".md", ".markdown"):
        parts = []
        for vnum, vtitle, vhtml in iter_volumes(mm_path, volumes=VOLUMES):
            parts.append(f"<h1 class='vol'>{html.escape(vtitle or ('Vol. %02d'%vnum))}</h1>{vhtml}")
        _write_pdf(_doc(_title_page("Multimodal catalogue"), "".join(parts)), out_path)
        print(f"multimodal PDF: {out_path}  ({os.path.getsize(out_path)//1024} KB)")
        return out_path
    rows = _mm_rows(mm_path)
    def volkey(r):
        try: return int(r.get("volume") or r.get("vol") or 0)
        except Exception: return 0
    rows.sort(key=lambda r: (volkey(r), str(r.get("scan", "")).zfill(4), str(r.get("region_uid", ""))))
    body, cur = [], None
    for r in rows:
        if VOLUMES is not None and volkey(r) not in VOLUMES: continue
        v = volkey(r)
        if v != cur:
            body.append(f"<h1 class='vol'>Vol. {v:02d}</h1>"); cur = v
        body.append(_mm_card(r))
    _write_pdf(_doc(_title_page("Multimodal catalogue · images · objects · inserts · marginalia"),
                    "".join(body)), out_path, base_url="/")
    print(f"multimodal PDF: {out_path}  ({len(rows)} regions, {os.path.getsize(out_path)//1024} KB)")
    return out_path

def _find_text_corpus():
    mds = [p for p in sorted(glob.glob(os.path.join(CORPUS_DIR, "*.md")))
           if "multimodal" not in os.path.basename(p).lower()]
    if not mds: return None
    named = [p for p in mds if "corpus" in os.path.basename(p).lower()]
    return (named or mds)[0]

def run():
    out_dir = OUT_DIR or CORPUS_DIR
    os.makedirs(out_dir, exist_ok=True)
    text_path = TEXT_CORPUS or _find_text_corpus()
    mm_candidates = sorted(glob.glob(os.path.join(CORPUS_DIR, "*multimodal*")))
    mm_path = MULTIMODAL or (mm_candidates[0] if mm_candidates else None)
    print("text corpus :", text_path)
    print("multimodal  :", mm_path)
    if text_path:
        render_text_pdf(text_path, os.path.join(out_dir, "Laubmann_corpus.pdf"),
                        volumes=VOLUMES, keep_parts=SPLIT_PER_VOLUME)
    if mm_path:
        render_multimodal_pdf(mm_path, os.path.join(out_dir, "Laubmann_multimodal.pdf"))


if __name__ == "__main__":
    run()

In [ ]:
!ls /content/drive/MyDrive/HistOrniGraph_output/corpus_2026-07-21/multimodal

In [ ]:
import pandas as pd
df = pd.read_csv(CORPUS_DIR / 'entries.csv')
print(len(df), 'entries across', df.volume.nunique(), 'volumes')
display(df.groupby('volume').size().rename('entries'))
print('\n--- coverage report (head) ---')
print((CORPUS_DIR / 'report.md').read_text(encoding='utf-8')[:1800])

In [ ]:
mm = pd.read_csv(CORPUS_DIR / 'multimodal' / 'multimodal.csv')
print(len(mm), 'non-text regions in the multimodal corpus')
display(mm.groupby('region_type').size().rename('regions'))
display(mm[['region_type','folded','entry_uid','description']].head(10))

## 6 · Is the Vol 1 → Vol 15 contamination still present?

You believe the source data was fixed. This checks the **freshly built** corpus
directly: it looks for the same `page_id` appearing in more than one volume
(the signature of a page reprocessed into another volume's output tree). If the
fix worked, this table is empty or tiny.


In [ ]:
import json
from collections import defaultdict

pages = json.load(open(CORPUS_DIR / 'corpus.json'))
by_pid = defaultdict(set)
for p in pages:
    by_pid[p['page_id']].add(int(p['volume']))

cross = {pid: sorted(v) for pid, v in by_pid.items() if len(v) > 1}
print(f'page_ids appearing in >1 volume: {len(cross)}')
if cross:
    rows = defaultdict(int)
    for pid, vols in cross.items():
        rows[tuple(vols)] += 1
    print('\nvolume-pair : shared page_ids')
    for vols, n in sorted(rows.items(), key=lambda x: -x[1]):
        print(f'  {vols} : {n}')
    print('\n→ Contamination is STILL present in this corpus. The dedup step below '
          'will cluster these by same-page-id-cross-volume; confirm & apply to remove them.')
else:
    print('\n→ Clean: no page_id spans multiple volumes. The Vol1/15 issue is not in this corpus.')

## 7 · Detect duplicates / near-duplicates

Writes a duplicates report + a transcription-quality report. `--image-root`
lets the perceptual-hash layer adjudicate the review band from the page crops.
Operating points: `--cluster-threshold 0.55` (review floor), `--high-threshold
0.80` (auto-drop band, ~0.97 precision).


In [ ]:
DEDUP_OUT = CORPUS_DIR / 'dedup'

!cd "{DEDUP}" && python detect_duplicates.py "{CORPUS_DIR}/corpus.json" \
    -o "{DEDUP_OUT}" \
    --scan-window 3 --cluster-threshold 0.55 --high-threshold 0.80 \
    --image-root "{OUTPUT_BASE}"

dup = pd.read_csv(DEDUP_OUT / 'duplicates_report.csv')
print(len(dup), 'clusters flagged')
display(dup.head(15))

## 8 · Review GUI → decisions

Download `review.html`, open it locally, confirm/reject each cluster and pick
the page to keep, then **Export decisions** → `dedup_decisions.json`. Upload
that JSON back to `DEDUP_OUT` on Drive for §9.


In [ ]:
!cd "{DEDUP}" && python build_review_gui.py "{DEDUP_OUT}/duplicates_report.jsonl" \
    --corpus "{CORPUS_DIR}/corpus.json" \
    -o "{DEDUP_OUT}/review.html"

from google.colab import files
files.download(str(DEDUP_OUT / 'review.html'))

## 9 · Apply decisions → clean corpus (non-destructive)

Writes a **new** `corpus_*_dedup/` dir with rebuilt `corpus.json` / `corpus.txt`
**and regenerated `entries.jsonl` / `entries.csv`** (the seam fix makes this
work), plus a `dedup_manifest` listing every dropped page and the page kept in
its place. The input corpus is untouched.


In [ ]:
DECISIONS    = DEDUP_OUT / 'dedup_decisions.json'      # exported from review.html, re-uploaded
CORPUS_DEDUP = CORPUS_DIR.parent / (CORPUS_DIR.name + '_dedup')

# apply_dedup.py adds the addons root to sys.path itself, so it resolves
# laubmann_corpus from any working directory — no extra setup needed.
!python "{DEDUP}/apply_dedup.py" \
    --corpus-dir "{CORPUS_DIR}" \
    --decisions  "{DECISIONS}" \
    --out-dir    "{CORPUS_DEDUP}"

print('\nClean corpus →', CORPUS_DEDUP)
import os
assert os.path.isfile(CORPUS_DEDUP / 'entries.csv'), 'entries not regenerated — check the import seam'
man = pd.read_csv(CORPUS_DEDUP / 'dedup_manifest.csv')
print(len(man), 'pages dropped')
display(man.head(15))

## 10 · Commit to git (only after it works)

Once you're happy with the outputs, commit the add-on code (not the Drive data)
into the HistOrniGraph repo. From your own machine, or from Colab:

```bash
# clone with a token or use the GitHub UI to add the folder
git -C HistOrniGraph checkout -b corpus-dedup
cp -r /content/HistOrniGraph_addons HistOrniGraph/
git -C HistOrniGraph add HistOrniGraph_addons
git -C HistOrniGraph commit -m "Add corpus builder refactor + multimodal + dedup toolset"
git -C HistOrniGraph push -u origin corpus-dedup
```

Then open a PR. The corpus artifacts themselves stay on Drive — only the code
goes in git.

### What this unblocks
- `corpus_*_dedup/` is the clean input for the **index-linking** (bird-index) and **knowledge-graph** (laubmann-kg) agents.
- The multimodal catalogue keyed by `entry_uid` becomes the KG's multimedia records.
- If §6 still shows contamination, that confirms the source-data fix didn't fully propagate — worth resolving upstream too, not only via dedup.
